# Python 기초 5 · 직접 풀어보는 연습문제 정답지

이 노트북은 `Python_Basic_5_Library_Overview_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 실습에 필요한 라이브러리 임포트와 사전 데이터를 정의합니다. 셀을 먼저 실행해주세요.
import math
import random
import json
from datetime import datetime, timedelta
from pathlib import Path

api_response_mock = '''{
    "status": "success",
    "timestamp": "2026-11-16T09:30:00Z",
    "payload": {
        "campaign_id": "CAMP_2026_Q4",
        "conversions": [
            {"user_id": 901, "revenue": 45000, "channel": "Google"},
            {"user_id": 902, "revenue": 128000, "channel": "Meta"},
            {"user_id": 903, "revenue": 67000, "channel": "Google"}
        ]
    }
}'''


## 문제 1. datetime을 활용한 고객 활동 기간 및 리텐션 일수 계산 [시계열 기초]

한 고객의 가입 일자와 최근 로그인 일자가 다음과 같은 문자열로 기록되어 있습니다:
- 가입 일자: `join_date_str = "2026-03-15"`
- 최근 로그인: `last_login_str = "2026-11-16"`

`datetime.strptime()`을 사용하여:
1. 두 문자열을 `datetime` 객체로 변환하세요.
2. 가입 후 최근 로그인까지 총 며칠(`days`)이 경과했는지 계산하세요.
3. 가입일로부터 정확히 100일째 되는 날(`milestone_100d`)의 날짜를 `YYYY-MM-DD` 형식으로 출력하세요 (`timedelta` 활용).


In [ ]:
# 문제 1 풀이
from datetime import datetime, timedelta

join_date_str = "2026-03-15"
last_login_str = "2026-11-16"

# 1. 문자열 -> datetime 객체 변환
join_date = datetime.strptime(join_date_str, "%Y-%m-%d")
last_login = datetime.strptime(last_login_str, "%Y-%m-%d")

# 2. 경과 일수 계산
elapsed_days = (last_login - join_date).days
print(f"가입 후 최근 로그인까지 경과 일수: {elapsed_days}일")

# 3. 100일 기념일 계산
milestone_100d = join_date + timedelta(days=100)
print(f"가입 100일 기념일: {milestone_100d.strftime('%Y-%m-%d')}")


**결과 해석 / 확인한 점:**  
1. **시계열 객체 연산의 장점:** 문자열 상태로는 월별 일수(30일, 31일, 윤달)를 직접 계산하기 극히 어렵지만, `datetime`과 `timedelta`를 쓰면 246일의 경과 일수 및 100일 경과 시점(2026-06-23)이 달력 규칙에 맞게 정확히 계산됩니다.
2. **리텐션 분석 연계:** 코호트(Cohort) 분석 시 가입 후 N일차(Day 1, Day 7, Day 30) 리텐션을 측정하는 표준 로직입니다.


## 문제 2. random.seed 고정을 통한 A/B 테스트 대조군/실험군 표본 추출 [실험 설계]

전체 1,000명의 활성 고객 풀(`user_pool = [f"U_{i:04d}" for i in range(1, 1001)]`)이 있습니다.
A/B 테스트를 위해 비복원 무작위 추출을 수행하려고 합니다.
- 요구사항:
  1. 실험의 재현성을 보장하기 위해 `random.seed(2026)`을 고정하세요.
  2. `random.sample()`을 사용하여 대조군(Control Group, A) 20명을 추출하세요.
  3. 전체 유저 풀에서 대조군을 제외한 나머지 유저들 중 실험군(Treatment Group, B) 20명을 추출하세요.
  4. 두 그룹 간 중복 유저가 없는지 세트 교집합(`set(group_a) & set(group_b)`)으로 검증하세요.


In [ ]:
# 문제 2 풀이
import random

user_pool = [f"U_{i:04d}" for i in range(1, 1001)]

# 1. 시드 고정
random.seed(2026)

# 2. 대조군(A) 20명 추출
group_a = random.sample(user_pool, 20)

# 3. 남은 유저 풀 구성 및 실험군(B) 20명 추출
remaining_pool = [u for u in user_pool if u not in group_a]
group_b = random.sample(remaining_pool, 20)

# 4. 상호 배타성(중복 없음) 검증
overlap = set(group_a) & set(group_b)

print("대조군(A) 앞 5명:", group_a[:5])
print("실험군(B) 앞 5명:", group_b[:5])
print("두 그룹 간 중복 고객 수:", len(overlap))
assert len(overlap) == 0, "중복 유저 발생 오류!"
print("검증 완료: 두 그룹은 완전히 독립적입니다.")


**결과 해석 / 확인한 점:**  
1. **재현성(Reproducibility) 확보:** `random.seed(2026)`을 선언함으로써 코드를 몇 번을 다시 실행해도 동일한 표본이 추출되어 데이터 과학 보고서의 신뢰성이 보장됩니다.
2. **독립 표본 추출:** 두 집단 간 교집합이 0건(`len(overlap) == 0`)임을 검증하여 신규 기능 배포 시 오염(Contamination) 없는 공정한 A/B 테스트 환경을 마련했습니다.


## 문제 3. pathlib.Path를 이용한 프로젝트 파일 및 데이터 자동 탐색 [디렉토리 자동화]

`pathlib.Path`는 운영체제에 구애받지 않고 파일 경로를 직관적으로 다룰 수 있는 현대적 표준 모듈입니다.
현재 프로젝트의 `Study_Python` 디렉토리(`/Users/a229/Documents/ChatGPT/Code_Study/Study_Python`) 내에서:
1. `Path` 객체로 디렉토리를 지정하고 실제 존재하는지(`.exists()`) 확인하세요.
2. `.glob("**/*.ipynb")`를 사용하여 해당 폴더 및 모든 하위 폴더에 있는 주피터 노트북(`.ipynb`) 파일들의 경로 목록을 검색하세요.
3. 검색된 전체 노트북 파일 개수와 앞 5개의 파일 이름을 출력하세요.


In [ ]:
# 문제 3 풀이
from pathlib import Path

target_dir = Path("/Users/a229/Documents/ChatGPT/Code_Study/Study_Python")

print("디렉토리 존재 여부:", target_dir.exists())

# 하위 폴더 전수 재귀 검색 (glob)
ipynb_files = list(target_dir.glob("**/*.ipynb"))

print(f"총 발견된 .ipynb 노트북 파일 개수: {len(ipynb_files)}개")
print("=== 상위 5개 노트북 파일명 ===")
for p in ipynb_files[:5]:
    print(f"- {p.name} (위치: {p.parent.name})")


**결과 해석 / 확인한 점:**  
1. **운영체제 중립적 경로 연산:** 슬래시(`/`) 연산자를 통해 경로를 결합하고, 윈도우와 맥/리눅스 간의 역슬래시(`\`) 문제를 완벽히 회피합니다.
2. **배치 파일 자동 수집:** `glob("**/*.ipynb")` 재귀 패턴을 통해 각 단원 폴더에 흩어져 있는 수십 개의 실습 및 교재 노트북을 코드 한 줄로 인덱싱할 수 있습니다.


## 문제 4. json 모듈을 활용한 REST API 로그 파싱 및 매출 집계 [반정형 데이터]

웹 서비스나 마케팅 광고 매체에서 전달된 `api_response_mock` JSON 문자열이 있습니다.
`json.loads()`를 사용하여:
1. JSON 문자열을 파이썬 딕셔너리로 역직렬화(Deserialize)하세요.
2. `payload` 내의 `conversions` 리스트를 추출하세요.
3. 채널이 `"Google"`인 주문들의 매출(`revenue`) 총합을 계산하여 출력하세요.


In [ ]:
# 문제 4 풀이
import json

# 1. JSON 문자열 -> 딕셔너리 파싱
data = json.loads(api_response_mock)

# 2. 전환 데이터 추출
conversions = data["payload"]["conversions"]
print(f"캠페인 ID: {data['payload']['campaign_id']}")
print(f"수신된 전환 건수: {len(conversions)}건")

# 3. Google 채널 매출 총합 집계
google_revenue = sum(c["revenue"] for c in conversions if c["channel"] == "Google")
print(f"Google 채널 발생 총 매출: {google_revenue:,}원")


**결과 해석 / 확인한 점:**  
1. **JSON과 파이썬 딕셔너리의 상호 변환:** 웹 표준 포맷인 JSON의 계층 구조가 파이썬의 중첩 딕셔너리/리스트로 자연스럽게 매핑됩니다.
2. **조건부 집계:** Google 채널을 통해 발생한 901번 주문(45,000원)과 903번 주문(67,000원)이 정확히 합산되어 총 112,000원의 매출이 집계되었습니다.


## 문제 5. math 모듈을 활용한 데이터 왜도(Skewness) 완화 로그 변환 [수학 변환]

이커머스 결제 데이터는 소수의 초고액 결제자로 인해 오른쪽 꼬리가 매우 긴 극단적 우측 왜도(Right Skewed)를 띱니다.
수치 안정화를 위해 `log1p(x) = log(1 + x)` 변환을 적용합니다.
- `skewed_revenues = [0, 15000, 32000, 48000, 12000000, 25000000]`

`math.log1p()` 함수를 사용하여:
1. 각 매출액에 자연로그 변환을 적용한 리스트 `log_transformed`를 생성하세요.
2. 원래 값과 로그 변환 후의 값(소수점 둘째 자리)을 나란히 비교 출력하세요.


In [ ]:
# 문제 5 풀이
import math

skewed_revenues = [0, 15000, 32000, 48000, 12000000, 25000000]

# 1. log1p 변환 (0원인 경우 log(1) = 0으로 안전하게 수렴)
log_transformed = [round(math.log1p(x), 2) for x in skewed_revenues]

print("=== 극단치 로그 변환 비교 ===")
for orig, transformed in zip(skewed_revenues, log_transformed):
    print(f"원래 매출: {orig:>10,}원  ->  로그 변환 값: {transformed:>6.2f}")


**결과 해석 / 확인한 점:**  
1. **log1p의 수학적 안정성:** 매출이 0원인 경우 일반 `math.log(0)`은 수학적으로 정의되지 않아 `ValueError`가 발생하지만, `math.log1p(x)`는 $\ln(1 + 0) = 0$으로 안전하게 변환됩니다.
2. **스케일 압축:** 0원에서 2,500만 원에 이르는 천문학적 격차가 0.00 ~ 17.03의 균일한 스케일로 압축되어 회귀 모델이나 신경망 학습의 안정성이 비약적으로 향상됩니다.


## 문제 6. 순수 파이썬 리스트 10만 개 연산 vs NumPy ndarray 고속화 벤치마킹 [성능 검증]

데이터 분석가가 왜 파이썬 기본 리스트를 넘어 NumPy와 Pandas를 필수적으로 배워야 하는지 입증하는 성능 측정 과제입니다.
- 요소 개수: `N = 100,000`
1. 파이썬 리스트 `[x * 1.1 for x in range(N)]`을 생성하는 소요 시간을 `time.perf_counter()`로 측정하세요.
2. 동일한 크기의 NumPy 배열 `np.arange(N) * 1.1` 연산 소요 시간을 측정하세요.
3. 두 연산의 소요 시간과 NumPy의 가속 배수(NumPy가 몇 배 더 빠른지)를 출력하세요.


In [ ]:
# 문제 6 풀이
import time
import numpy as np

N = 100000

# 1. 순수 파이썬 리스트 컴프리헨션 측정
t0 = time.perf_counter()
py_result = [x * 1.1 for x in range(N)]
t1 = time.perf_counter()
py_time = t1 - t0

# 2. NumPy 벡터화 연산 측정
t2 = time.perf_counter()
np_result = np.arange(N) * 1.1
t3 = time.perf_counter()
np_time = t3 - t2

speedup = py_time / np_time if np_time > 0 else 1.0

print(f"데이터 크기: {N:,}건")
print(f"파이썬 리스트 연산 시간: {py_time:.6f}초")
print(f"NumPy 배열 연산 시간:   {np_time:.6f}초")
print(f"NumPy 연산 가속 배수:     약 {speedup:.1f}배 빠름!")


**결과 해석 / 확인한 점:**  
1. **C 언어 기반 벡터화의 우월성:** 파이썬 리스트는 객체 참조 포인터의 배열이어서 반복문마다 타입 체크와 메모리 점프가 발생하지만, NumPy `ndarray`는 동일 타입 수치가 메모리에 연속적으로 배치(Contiguous Memory Layout)되어 SIMD 및 캐시 효율을 극대화합니다.
2. **커리큘럼 연계:** 이로써 파이썬 기본 문법(Stage 1)에서 다음 단계인 고성능 수치 연산(Stage 2: NumPy)과 표 데이터 분석(Pandas)으로 나아가야 하는 필연적인 이유를 실측 성능으로 입증했습니다.
